In [1]:
import os
import json
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# ============================ 全局配置 ============================
MODEL_PATH = "./best_model_gen_05"
QSA_DATA_PATH = "PandaAIQ.jsonl"
MBPP_DATA_PATH = "mbpp.jsonl"
# 目标干预层
TARGET_LAYERS = [-5, -10, -15]
# 消融系数序列
ABLATION_RATIOS = [0, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0]

def set_academic_style():
    plt.rcParams.update({'font.family': 'serif', 'font.size': 10, 'axes.grid': True})

# ============================ 工具函数：提取代码块 ============================
def extract_code(text):
    import re
    code = re.findall(r'```python\n(.*?)\n```', text, re.DOTALL)
    if code:
        return code[-1]
    code = re.findall(r'```\n(.*?)\n```', text, re.DOTALL)
    if code:
        return code[-1]
    return text.strip()

# ============================ 错误分类逻辑 ============================
def classify_error(response, is_qsa):
    res = response.lower()
    if is_qsa:
        # QSA 身份识别任务判定
        has_identity = any(kw in res for kw in ["panda", "ai", "assistant"])
        if has_identity:
            return "Correct/Partial"
        if len(res.split()) < 3 or res.count(res.split()[0] if res.split() else "") > 2:
            return "Degraded (Language Collapse)"
        return "Identity Loss (Generic Assistant)"
    else:
        # MBPP 代码生成任务判定
        code = extract_code(response)
        if 'def ' in code and len(code) > 30:
            return "Correct"
        if len(response.strip()) < 5 or "sorry" in res or "i don't know" in res:
            return "Logic Failure (No Valid Code)"
        return "Incorrect Code"

# ============================ 激活值提取函数 ============================
def get_activations(model, tokenizer, items, layers_indices, is_qsa_data=False):
    print("👉 开始提取激活值...")
    all_layer_acts = {idx: [] for idx in layers_indices}
    real_device = next(model.parameters()).device

    def get_hook(idx):
        def hook(module, input, output):
            val = output[0] if isinstance(output, tuple) else output
            all_layer_acts[idx].append(val[:, -1, :].detach().to(torch.float32).cpu().numpy())
        return hook

    handles = [model.model.layers[idx].register_forward_hook(get_hook(idx)) for idx in layers_indices]
    model.eval()
    with torch.inference_mode():
        for item in tqdm(items, desc="Extract Acts"):
            if is_qsa_data:
                q_text = item.get("question", "")
                prompt = f"Question: {q_text}\nAnswer:"
            else:
                m_text = item.get("text", "")
                prompt = f"Question: {m_text}\nWrite a function to solve this problem.\nAnswer:"
            inputs = tokenizer(prompt, return_tensors="pt").to(real_device)
            model(**inputs)
    for h in handles:
        h.remove()
    print("✅ 激活值提取完成")
    return {idx: np.concatenate(acts, axis=0) for idx, acts in all_layer_acts.items()}

# ============================ 消融实验主函数 ============================
def run_comprehensive_ablation(model, tokenizer, dataset, layers_indices, directions_dict, ratio, is_qsa=False):
    print(f"--- 开始消融实验 ratio={ratio}, is_qsa={is_qsa} ---")
    correct = 0
    total_kl = 0.0
    error_stats = {"Correct": 0, "Identity_Loss": 0, "Logic_Failure": 0, "Degraded": 0}
    real_device = next(model.parameters()).device

    # 层激活干预Hook
    def get_hard_hook(direction_vec):
        dir_vec = direction_vec.to(real_device).to(torch.float32).view(-1, 1)
        def hook(module, input, output):
            val = output[0] if isinstance(output, tuple) else output
            val_f = val.to(torch.float32)
            proj = torch.matmul(torch.matmul(val_f, dir_vec), dir_vec.transpose(0, 1))
            res = val_f - ratio * proj
            return (res.to(val.dtype),) if isinstance(output, tuple) else res.to(val.dtype)
        return hook

    # 预计算原始logits（用于KL散度计算）
    print("预计算原始logits...")
    all_orig_logits = []
    with torch.inference_mode():
        for item in tqdm(dataset, desc="Origin Logits"):
            if is_qsa:
                q_text = item.get("question", "")
                prompt = f"Question: {q_text}\nAnswer:"
            else:
                m_text = item.get("text", "")
                prompt = f"Question: {m_text}\nWrite a function to solve this problem.\nAnswer:"
            inputs = tokenizer(prompt, return_tensors="pt").to(real_device)
            logits = model(**inputs).logits[:, -1, :].detach()
            all_orig_logits.append(logits)

    # 注册干预Hook
    handles = [model.model.layers[idx].register_forward_hook(get_hard_hook(directions_dict[idx])) for idx in layers_indices]
    print("已注册干预Hook，开始推理生成...")

    with torch.inference_mode():
        for i, item in enumerate(tqdm(dataset, desc="Infer & Generate")):
            if is_qsa:
                q_text = item.get("question", "")
                prompt = f"Question: {q_text}\nAnswer:"
            else:
                m_text = item.get("text", "")
                prompt = f"Question: {m_text}\nWrite a function to solve this problem.\nAnswer:"
            inputs = tokenizer(prompt, return_tensors="pt").to(real_device)
            out_logits = model(**inputs).logits[:, -1, :]

            # 计算KL散度
            kl = F.kl_div(
                F.log_softmax(out_logits, dim=-1),
                F.softmax(all_orig_logits[i], dim=-1),
                reduction='batchmean'
            ).item()
            total_kl += kl

            # 文本/代码生成（消除采样参数警告，保证代码完整）
            gen = model.generate(
                **inputs,
                max_new_tokens=256,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False,
                num_beams=1,
                temperature=None,
                top_p=None,
                top_k=None
            )
            res = tokenizer.decode(gen[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

            # 分类错误类型
            err_type = classify_error(res, is_qsa)
            if "Correct" in err_type:
                correct += 1
                error_stats["Correct"] += 1
            if "Identity Loss" in err_type:
                error_stats["Identity_Loss"] += 1
            if "Logic Failure" in err_type:
                error_stats["Logic_Failure"] += 1
            if "Degraded" in err_type or "Language Collapse" in err_type:
                error_stats["Degraded"] += 1

    # 移除Hook + 释放显存碎片
    for h in handles:
        h.remove()
    torch.cuda.empty_cache()
    print(f"--- ratio={ratio} 实验结束 ---")
    return (correct / len(dataset)) * 100, (total_kl / len(dataset)), error_stats

# ============================ 主程序入口 ============================
def main():
    set_academic_style()
    print("加载模型与分词器...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    print("模型加载完成")

    # 加载 QSA 全量实验数据（取前100条）
    with open(QSA_DATA_PATH, 'r', encoding="utf-8") as f:
        qsa_data = [json.loads(line) for line in f][:100]

    # 加载 MBPP 全量实验数据（取前100条）
    with open(MBPP_DATA_PATH, 'r', encoding="utf-8") as f:
        mbpp_data = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            mbpp_data.append({
                "text": item.get("text", ""),
                "code": item.get("code", ""),
                "test_list": item.get("test_list", [])
            })
    mbpp_data = mbpp_data[:100]

    # 用前50条样本计算方向向量（激活提取 + PCA）
    q_acts = get_activations(model, tokenizer, qsa_data[:50], TARGET_LAYERS, is_qsa_data=True)
    m_acts = get_activations(model, tokenizer, mbpp_data[:50], TARGET_LAYERS, is_qsa_data=False)
    directions = {}

    for idx in TARGET_LAYERS:
        # 正式版：PCA维度恢复为20（50条样本，合法）
        pca_m = PCA(n_components=20).fit(m_acts[idx])
        diff = q_acts[idx] - m_acts[idx].mean(axis=0)
        q_dir = PCA(n_components=1).fit(diff).components_[0]
        # 正交去除MBPP主成分
        for s_vec in pca_m.components_:
            q_dir -= np.dot(q_dir, s_vec) * s_vec
        directions[idx] = torch.tensor(q_dir / np.linalg.norm(q_dir))

    # 遍历所有消融系数，执行完整实验
    full_results = []
    prev_q_kl, prev_m_kl = 0.0, 0.0

    for r in tqdm(ABLATION_RATIOS, desc="Full Analysis (QSA vs MBPP)"):
        q_acc, q_kl, q_err = run_comprehensive_ablation(
            model, tokenizer, qsa_data, TARGET_LAYERS, directions, r, is_qsa=True
        )
        m_acc, m_kl, m_err = run_comprehensive_ablation(
            model, tokenizer, mbpp_data, TARGET_LAYERS, directions, r, is_qsa=False
        )

        # 计算KL敏感度（斜率）
        q_sens = (q_kl - prev_q_kl) / (r if r > 0 else 1)
        m_sens = (m_kl - prev_m_kl) / (r if r > 0 else 1)

        full_results.append({
            "Ratio": r,
            "QSA_Accuracy": q_acc,
            "QSA_KL": q_kl,
            "QSA_Sensitivity": q_sens,
            "QSA_IdentityLoss": q_err["Identity_Loss"],
            "QSA_Degraded": q_err["Degraded"],

            "MBPP_Accuracy": m_acc,
            "MBPP_KL": m_kl,
            "MBPP_Sensitivity": m_sens,
            "MBPP_LogicFailure": m_err["Logic_Failure"],
            "MBPP_Degraded": m_err["Degraded"]
        })
        prev_q_kl, prev_m_kl = q_kl, m_kl

    # 导出实验结果至Excel
    df = pd.DataFrame(full_results)
    df.to_excel("ablation_results_qsa_mbpp_formal.xlsx", index=False)
    print("✅ 正式实验结果已保存至 ablation_results_qsa_mbpp_formal.xlsx")

    # ========== 如需绘制对比图，取消下面注释即可 ==========
    # plt.figure(figsize=(8, 6))
    # plt.plot(df['Ratio'], df['QSA_Sensitivity'], 'r-o', label='QSA Sensitivity (Identity)')
    # plt.plot(df['Ratio'], df['MBPP_Sensitivity'], 'b--s', label='MBPP Sensitivity (Code Generation)')
    # plt.title("Sensitivity Ratio Comparison: QSA vs MBPP")
    # plt.xlabel("Ablation Ratio")
    # plt.ylabel("d(KL) / d(Ratio)")
    # plt.legend()
    # plt.tight_layout()
    # plt.savefig("Sensitivity_QSA_MBPP_formal.png")
    # plt.show()

if __name__ == "__main__":
    main()

加载模型与分词器...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

模型加载完成
👉 开始提取激活值...


Extract Acts: 100%|██████████| 50/50 [00:01<00:00, 32.76it/s]


✅ 激活值提取完成
👉 开始提取激活值...


...

--- ratio=5.0 实验结束 ---
--- 开始消融实验 ratio=5.0, is_qsa=False ---
预计算原始logits...



Origin Logits: 100%|██████████| 100/100 [00:02<00:00, 46.60it/s]


已注册干预Hook，开始推理生成...



Full Analysis (QSA vs MBPP): 100%|██████████| 7/7 [2:02:09<00:00, 1047.14s/it]

--- ratio=5.0 实验结束 ---
✅ 正式实验结果已保存至 ablation_results_qsa_mbpp_formal.xlsx
